# Fine-tune **Qwen2.5-VL-7B** for TTB Label Verification

Self-contained Colab notebook. Trains a LoRA adapter on top of `unsloth/Qwen2.5-VL-7B-Instruct` using Unsloth, on data pulled from the COLA Cloud free sample (CC0). Outputs go to your Google Drive at `/MyDrive/ttb_sft/qwen2.5-vl-7b/`.

**What to do before running**:
1. `Runtime → Change runtime type → GPU` (A100 ideal; T4 works for the small Kimi-VL but Qwen2.5-VL-7B needs 24GB+).
2. `Runtime → Run all`. The notebook is designed to run end-to-end without intervention.
3. Caffeinate your Mac / leave the tab open. On a Colab A100 the full run is **~3–6 hours**.

**What it does**:
- Pulls 1,000 COLA Cloud records + ~800 label images
- Builds a structured JSON ground-truth target per image from the dataset's `BRAND_NAME`, `CLASS_NAME`, `PRODUCT_NAME`, `OCR_ABV`, `OCR_VOLUME`, `ORIGIN_NAME`, and `OCR_TEXT` columns
- 80/10/10 train/val/test split
- LoRA fine-tune (rank 16, attention projections)
- Per-field extraction accuracy on the held-out test set
- Writes the adapter + tokenizer config + eval report to Drive

**Note on TTB data.gov**: the data.gov 'Public COLA Registry Search and Download' resource is a link to the interactive UI at `ttbonline.gov` — there's no public bulk CSV. The COLA Cloud free sample (also sourced from the TTB registry) is the only direct-fetch path. If you later want to scrape additional COLAs by TTB_ID, that's a separate pull job; the notebook architecture supports adding more (image, JSON) pairs to the dataset dir without code changes.


## 0. Install dependencies & set model tag


In [ ]:
!pip install -q --upgrade pip
!pip install -q "unsloth[colab-new]" 'trl<0.10.0' peft accelerate bitsandbytes
!pip install -q datasets pillow requests tqdm


In [ ]:
MODEL_TAG = 'qwen2_5_vl_7b'
BASE_MODEL = 'unsloth/Qwen2.5-VL-7B-Instruct'


In [ ]:
# Mount Drive — adapter + checkpoints persist between runs.
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
MODEL_TAG = MODEL_TAG  # set in the previous cell
WORK_DIR = pathlib.Path(f'/content/drive/MyDrive/ttb_sft/{MODEL_TAG}')
WORK_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = pathlib.Path('/content/cola_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
IMG_DIR  = DATA_DIR / 'images'; IMG_DIR.mkdir(exist_ok=True)
print('Work dir:', WORK_DIR)
print('Data dir:', DATA_DIR)


## 1. Download COLA Cloud free sample

1,000 recent COLA records + ~1,750 image rows. ~500 KB zip. CC0 license.


In [ ]:
import urllib.request, zipfile, os
COLA_ZIP_URL = 'https://dyuie4zgfxmt6.cloudfront.net/samples/cola-sample-pack-v1.zip'
zip_path = DATA_DIR / 'sample.zip'
if not (DATA_DIR / 'cola.csv').exists():
    print('Downloading COLA Cloud sample...')
    urllib.request.urlretrieve(COLA_ZIP_URL, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    print('  Extracted.')
else:
    print('Already downloaded.')
!ls -la {DATA_DIR}


## 2. Build (image, structured-JSON) training pairs

For each label image where the dataset has usable metadata, we synthesize the structured JSON the model should learn to produce. The schema matches the backend's `ExtractedLabel` shape so the SFT'd model is drop-in for `INFERENCE_MODE=sft`.

Ground-truth labels come from the COLA Cloud dataset itself:
- `BRAND_NAME`, `PRODUCT_NAME`, `CLASS_NAME`, `ORIGIN_NAME` — directly from the COLA form
- `OCR_ABV`, `OCR_VOLUME` — Google Vision OCR run on the label image
- `OCR_TEXT` — full per-image OCR (used to extract the warning text + bottler line)

Images that don't have a downloadable file or don't have `GOVERNMENT WARNING` in the OCR are dropped (they don't help the warning-verification task).


In [ ]:
import csv, json, re
from collections import defaultdict
from pathlib import Path

VERBATIM_GOVERNMENT_WARNING = (
    'GOVERNMENT WARNING: (1) According to the Surgeon General, women should not '
    'drink alcoholic beverages during pregnancy because of the risk of birth defects. '
    '(2) Consumption of alcoholic beverages impairs your ability to drive a car or '
    'operate machinery, and may cause health problems.'
)

def _net_str(row):
    v, u = (row.get('OCR_VOLUME') or '').strip(), (row.get('OCR_VOLUME_UNIT') or '').strip().lower()
    if not v: return ''
    try:
        f = float(v); v = str(int(f)) if f.is_integer() else str(f)
    except ValueError: pass
    unit_map = {'milliliters':'mL','milliliter':'mL','ml':'mL',
                'liters':'L','liter':'L','l':'L',
                'fluid ounces':'FL OZ','fluid ounce':'FL OZ','fl oz':'FL OZ','fl. oz.':'FL OZ',
                'ounces':'FL OZ','ounce':'FL OZ','oz':'FL OZ'}
    return f"{v} {unit_map.get(u, u or 'mL')}"

def _abv_str(row):
    v = (row.get('OCR_ABV') or '').strip()
    if not v: return ''
    try:
        f = float(v); return f'{f:g}% Alc./Vol.'
    except ValueError: return ''

def _extract_warning(ocr_text):
    if not ocr_text or 'GOVERNMENT WARNING' not in ocr_text.upper():
        return None
    # take from 'GOVERNMENT WARNING' to end of 'health problems.'
    start = ocr_text.upper().find('GOVERNMENT WARNING')
    tail = ocr_text[start:]
    m = re.search(r'machinery[^.]*\.\s*', tail, re.IGNORECASE)
    return tail[:m.end()].strip() if m else tail[:500].strip()

def _extract_bottler(ocr_text):
    if not ocr_text: return ''
    for kw in ('BOTTLED BY', 'BREWED BY', 'PRODUCED BY', 'DISTILLED BY', 'IMPORTED BY',
               'Bottled by', 'Brewed by', 'Produced by', 'Distilled by', 'Imported by'):
        if kw in ocr_text:
            idx = ocr_text.find(kw)
            end = ocr_text.find('.', idx)
            return ocr_text[idx:end if end != -1 else idx+200].strip()
    return ''

def _derive_beverage(row):
    pt = (row.get('PRODUCT_TYPE') or '').lower(); cn = (row.get('CLASS_NAME') or '').lower()
    if 'seltzer' in cn or 'flavored' in cn: return 'malt'
    if pt == 'distilled spirits': return 'spirits'
    if pt == 'wine': return 'wine'
    if pt == 'malt beverage': return 'beer'
    return 'wine'

# Load CSVs
cola = list(csv.DictReader(open(DATA_DIR / 'cola.csv')))
imgs = list(csv.DictReader(open(DATA_DIR / 'cola_image.csv')))
by_ttb = defaultdict(list)
for ir in imgs: by_ttb[ir['TTB_ID']].append(ir)

# Build candidate pairs
candidates = []
for cola_row in cola:
    ttb_id = cola_row['TTB_ID']
    images_for = by_ttb.get(ttb_id, [])
    if not images_for: continue
    for img_row in images_for:
        ocr_text = img_row.get('OCR_TEXT') or ''
        if not ocr_text: continue
        warning_text = _extract_warning(ocr_text)
        target = {
            'fields': {},
            'government_warning': {
                'present': warning_text is not None,
                'detected_text': warning_text or '',
                'casing_all_caps': True if warning_text and warning_text[:18].isupper() else False,
                'heading_bold': True if warning_text else False,
                'body_bold': False,
                'approx_font_mm': None,
                'contrast_ok': True,
                'separate_and_apart': True,
            },
            'image_quality': {'score': 0.85, 'legible': True, 'note': None},
        }
        # Field labels (only when populated AND the field is likely on this panel)
        pos = img_row.get('CONTAINER_POSITION','').upper()
        is_back = pos == 'BACK'
        # Brand / class / product appear on FRONT typically
        if not is_back:
            if cola_row.get('BRAND_NAME'):
                target['fields']['Brand name'] = {'value': cola_row['BRAND_NAME'], 'confidence': 0.95}
            if cola_row.get('CLASS_NAME'):
                target['fields']['Class & type'] = {'value': cola_row['CLASS_NAME'], 'confidence': 0.95}
        # ABV / Net contents — OCR sourced; may be either panel
        if _abv_str(cola_row) and img_row.get('TTB_IMAGE_ID') == cola_row.get('OCR_ABV_TTB_IMAGE_ID'):
            target['fields']['Alcohol content'] = {'value': _abv_str(cola_row), 'confidence': 0.92}
        if _net_str(cola_row) and img_row.get('TTB_IMAGE_ID') == cola_row.get('OCR_VOLUME_TTB_IMAGE_ID'):
            target['fields']['Net contents'] = {'value': _net_str(cola_row), 'confidence': 0.92}
        # Bottler — usually on back
        bottler = _extract_bottler(ocr_text)
        if bottler and is_back:
            target['fields']['Bottler name/address'] = {'value': bottler, 'confidence': 0.85}
        # Country of origin — imports only, usually on back
        if (cola_row.get('DOMESTIC_OR_IMPORTED') or '').lower() == 'imported' and cola_row.get('ORIGIN_NAME') and is_back:
            target['fields']['Country of origin'] = {'value': f"Product of {cola_row['ORIGIN_NAME'].title()}", 'confidence': 0.85}
        # Only keep examples with at least 1 field OR the warning
        if not target['fields'] and not target['government_warning']['present']:
            continue
        candidates.append({
            'ttb_image_id': img_row['TTB_IMAGE_ID'],
            'ttb_id': ttb_id,
            'container_position': pos,
            'beverage_type': _derive_beverage(cola_row),
            'target': target,
        })

print(f'Built {len(candidates)} candidate (image, JSON) pairs')
print('Sample target JSON:')
print(json.dumps(candidates[0]['target'], indent=2)[:800])


## 3. Download label images (parallel, ~2–5 minutes)


In [ ]:
import concurrent.futures, urllib.request
CDN = 'https://dyuie4zgfxmt6.cloudfront.net/{}.webp'

def _download_one(cand):
    ttb_image_id = cand['ttb_image_id']
    dest = IMG_DIR / f'{ttb_image_id}.webp'
    if dest.exists() and dest.stat().st_size > 0:
        return cand, True
    try:
        req = urllib.request.Request(CDN.format(ttb_image_id), headers={'User-Agent': 'ttb-sft/0.1'})
        with urllib.request.urlopen(req, timeout=20) as r:
            data = r.read()
        dest.write_bytes(data)
        return cand, True
    except Exception as e:
        return cand, False

downloaded = []
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as ex:
    for cand, ok in ex.map(_download_one, candidates):
        if ok:
            cand['image_path'] = str(IMG_DIR / f'{cand["ttb_image_id"]}.webp')
            downloaded.append(cand)

print(f'Downloaded {len(downloaded)} / {len(candidates)} images')


## 4. Train / val / test split (80 / 10 / 10) + sanity check


In [ ]:
import random
random.seed(42)
random.shuffle(downloaded)
n = len(downloaded)
n_train = int(n * 0.80); n_val = int(n * 0.10)
train = downloaded[:n_train]
val   = downloaded[n_train:n_train + n_val]
test  = downloaded[n_train + n_val:]
print(f'train={len(train)}  val={len(val)}  test={len(test)}')

# Persist the split so we can re-eval against the same test set later.
import json
(WORK_DIR / 'splits').mkdir(exist_ok=True)
for name, split in [('train', train), ('val', val), ('test', test)]:
    with open(WORK_DIR / 'splits' / f'{name}.jsonl', 'w') as f:
        for ex in split:
            f.write(json.dumps({k: v for k, v in ex.items() if k != 'target'}) + '\n')
            # store target separately so the file is parsable for eval
    print(f'  wrote {name} split → {WORK_DIR}/splits/{name}.jsonl')


In [ ]:
SYSTEM_PROMPT = (
    'You are a careful transcription assistant for U.S. TTB alcohol label review. '
    'Given ONE label panel image and the beverage type, you READ what is printed and '
    'RETURN ONLY a JSON object matching the schema below. Do NOT decide whether the '
    'label complies with regulations.\n\n'
    'If a field is not clearly visible on the panel, OMIT it from the fields object. '
    'Never guess; never substitute deposit codes (e.g. "CA CRV"), NOM IDs, or barcodes. '
    'Transcribe the Government Warning EXACTLY as printed (preserve case, punctuation, '
    'any wording errors).\n\n'
    'Schema: {fields: {<field name>: {value, confidence}}, government_warning: '
    '{present, detected_text, casing_all_caps, heading_bold, body_bold, approx_font_mm, '
    'contrast_ok, separate_and_apart}, image_quality: {score, legible, note}}'
)


## 5. Load Qwen2.5-VL-7B with Unsloth (4-bit + LoRA)


In [ ]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    'unsloth/Qwen2.5-VL-7B-Instruct',
    load_in_4bit = True,
    use_gradient_checkpointing = 'unsloth',
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # keep vision tower frozen; train projection + LM
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = 'none',
    random_state = 42,
    target_modules = 'all-linear',
)
print(model.print_trainable_parameters())


## 6. Format dataset for Unsloth SFT trainer


In [ ]:
from PIL import Image
from unsloth.trainer import UnslothVisionDataCollator

def _convo(ex):
    return {
        'messages': [
            {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
            {'role': 'user',   'content': [
                {'type': 'image', 'image': Image.open(ex['image_path']).convert('RGB')},
                {'type': 'text',  'text': f"Beverage type: {ex['beverage_type']}. Extract per the schema."},
            ]},
            {'role': 'assistant', 'content': [
                {'type': 'text', 'text': json.dumps(ex['target'])},
            ]},
        ],
    }

train_ds = [_convo(ex) for ex in train]
val_ds   = [_convo(ex) for ex in val[:32]]   # cap val to keep eval fast during training
print('train examples:', len(train_ds), 'val examples:', len(val_ds))


## 7. Train — Unsloth SFT trainer


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_ds,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        logging_steps = 5,
        optim = 'adamw_8bit',
        weight_decay = 0.01,
        lr_scheduler_type = 'linear',
        seed = 42,
        output_dir = str(WORK_DIR / 'checkpoints'),
        save_strategy = 'epoch',
        report_to = 'none',
        remove_unused_columns = False,
        dataset_text_field = '',
        dataset_kwargs = {'skip_prepare_dataset': True},
        dataset_num_proc = 1,
        max_seq_length = 2048,
    ),
)
trainer_stats = trainer.train()
print('Done:', trainer_stats)


## 8. Evaluate on the held-out test set

Per-field accuracy + bucket distribution. The eval mirrors the backend's rules engine to give a head-to-head against Claude's numbers.


In [ ]:
import json, re
from collections import Counter

def _norm(s):
    s = (s or '').strip().lower()
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'[.,]', '', s)
    return s

def _generate_json(image_path, beverage_type):
    """Run the trained model on one image and return its parsed JSON."""
    from PIL import Image
    img = Image.open(image_path).convert('RGB')
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': [
            {'type': 'image', 'image': img},
            {'type': 'text', 'text': f'Beverage type: {beverage_type}. Extract per the schema.'},
        ]},
    ]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    txt = processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0]
    # Try to find the JSON object in the output
    try:
        start = txt.find('{')
        return json.loads(txt[start:txt.rfind('}') + 1])
    except Exception:
        return None

field_correct = Counter(); field_total = Counter()
warning_present_match = 0; warning_total = 0
n_parsed = n_unparseable = 0

from tqdm.auto import tqdm
for ex in tqdm(test, desc='Evaluating'):
    pred = _generate_json(ex['image_path'], ex['beverage_type'])
    if pred is None:
        n_unparseable += 1
        continue
    n_parsed += 1
    # Per-field accuracy
    for fname, fobs in (ex['target']['fields'] or {}).items():
        field_total[fname] += 1
        got = (pred.get('fields') or {}).get(fname, {}).get('value', '')
        if _norm(got) == _norm(fobs['value']):
            field_correct[fname] += 1
    # Warning presence + verbatim closeness
    gt_w = ex['target']['government_warning']
    pred_w = pred.get('government_warning') or {}
    warning_total += 1
    if bool(gt_w.get('present')) == bool(pred_w.get('present')):
        warning_present_match += 1

print(f'\nParsed: {n_parsed} / Unparseable: {n_unparseable}')
print('Per-field accuracy:')
for fname in sorted(field_total):
    acc = field_correct[fname] / field_total[fname]
    print(f'  {fname:<24} {field_correct[fname]:>3} / {field_total[fname]:>3} = {acc:.1%}')
print(f'Warning-presence accuracy: {warning_present_match} / {warning_total} = {warning_present_match/warning_total:.1%}')

# Save report to Drive
report = {
    'n_test': len(test), 'n_parsed': n_parsed, 'n_unparseable': n_unparseable,
    'field_accuracy': {f: {'correct': field_correct[f], 'total': field_total[f]} for f in field_total},
    'warning_present_accuracy': warning_present_match / warning_total if warning_total else None,
}
with open(WORK_DIR / 'eval_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print(f'\nReport written to {WORK_DIR}/eval_report.json')


## 9. Save LoRA adapter to Drive (downloadable, integrates with backend)


In [ ]:
import shutil, json
ADAPTER_DIR = WORK_DIR / 'adapter'
ADAPTER_DIR.mkdir(exist_ok=True)

# Save adapter
model.save_pretrained(str(ADAPTER_DIR))
# Save processor (so the backend can re-load with the same tokenizer config)
try:
    tokenizer.save_pretrained(str(ADAPTER_DIR))
except Exception:
    processor.save_pretrained(str(ADAPTER_DIR))

# Write a small manifest so the backend knows what to load
manifest = {
    'model_tag': MODEL_TAG,
    'base_model': BASE_MODEL,
    'trained_on': 'COLA Cloud free sample (CC0) — N=' + str(len(train)),
    'lora_rank': 16,
    'epochs': 3,
}
with open(ADAPTER_DIR / 'manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'Adapter + processor saved to {ADAPTER_DIR}')
print('Download it via Files panel → drive/MyDrive/ttb_sft/' + MODEL_TAG + '/adapter')


## 10. Zip the adapter and download to your Mac

Bundles the adapter + processor config + eval report into a single zip
and triggers a browser download to your default Downloads folder. After
this cell finishes, the file `~/Downloads/ttb_sft_{model_tag}.zip` is on
your Mac, ready to integrate into the backend.


In [ ]:
import shutil
from google.colab import files

# Include adapter + splits + eval report in the bundle so we can audit later.
BUNDLE_DIR = WORK_DIR / f'ttb_sft_{MODEL_TAG}'
BUNDLE_DIR.mkdir(exist_ok=True)
import os
for src in ['adapter', 'splits', 'eval_report.json']:
    src_path = WORK_DIR / src
    if src_path.exists():
        dst = BUNDLE_DIR / src
        if dst.exists():
            shutil.rmtree(dst) if dst.is_dir() else dst.unlink()
        if src_path.is_dir():
            shutil.copytree(src_path, dst)
        else:
            shutil.copy(src_path, dst)

zip_base = str(WORK_DIR / f'ttb_sft_{MODEL_TAG}')
zip_path = shutil.make_archive(zip_base, 'zip', root_dir=str(BUNDLE_DIR))
size_mb = os.path.getsize(zip_path) / 1024**2
print(f'Bundle: {zip_path}  ({size_mb:.1f} MB)')
print('Triggering browser download — accept the prompt that pops up.')
files.download(zip_path)


## 11. Integrate the trained model into the backend

After this notebook finishes, the LoRA adapter sits in your Drive at:
```
/MyDrive/ttb_sft/{model_tag}/adapter/
```

Local-side steps (next session):
1. Download the `adapter/` directory to `backend/models/{model_tag}/`.
2. Add a new `INFERENCE_MODE=sft` adapter in `backend/app/extractors/sft.py` that loads the base model + LoRA via vLLM (production) or transformers (dev) and returns the same `ExtractedLabel` shape.
3. Register the mode in `backend/app/extractors/factory.py`.
4. Run `make serve-sft` (we'll add it to the Makefile).
5. Re-run the eval against the same test set — head-to-head with Claude.

If the eval beats Claude on per-field accuracy AND latency is < 2s on a single GPU, this becomes the recommended path for the production demo.
